# C2: five-way comparison and model training (SRCD)

This notebook answers the C2 plan's evaluation question: does comparing each person with **their own normal movement** (SRCD) detect facial grimacing (FG) and head movement (HM) at stutter moments better than the alternatives?

| # | Approach | Trained? |
|---|---|---|
| 0 | Majority class (always the training fold's most common answer) | no: the floor every method must beat |
| 1 | Fixed-threshold rules (the earlier prototype) | no |
| 2 | Random Forest on 2-second window statistics | yes |
| 3 | GRU **without** the self-comparison step (population-standardised input) | yes |
| 4a | SRCD, label-free (the service's default) | no |
| 4b | SRCD + GRU (the team's model) | yes |
| 5 | CASA's published video model | reference numbers only (different test set) |

Every approach is scored on **the same labelled moments**, with **participant-grouped folds**: no participant is in both training and test.

**Metrics, per class (FG, HM):** accuracy next to the majority-class accuracy, precision, recall, F1 and PR-AUC (only for methods that give a score). Accuracy alone is misleading here: most moments have no behaviour, so "always no" already scores high.

**Thresholds** of the trained methods are tuned on a **validation split of the training participants** (never the test fold). A class whose cross-validated precision is below `SWARA_C2_MIN_PRECISION` (0.5), or whose accuracy does not beat the majority class, is reported as **not assessed**; the saved model records it and the service then uses label-free SRCD for that class.

**Before running:** build the dataset with `training/build_dataset.py` (see `notebooks/README.md`). Settings come from environment variables, so the same notebook runs a quick smoke test or the full study.

In [1]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # the vision service's own code (app/, training/)

DATA = Path(os.environ.get("SWARA_C2_DATA", ROOT / "data" / "casa.npz"))  # from training/build_dataset.py
FOLDS = int(os.environ.get("SWARA_C2_FOLDS", 5))
EPOCHS = int(os.environ.get("SWARA_C2_EPOCHS", 60))
SEED = int(os.environ.get("SWARA_C2_SEED", 17))
SAVE_MODEL = os.environ.get("SWARA_C2_SAVE_MODEL", "1") == "1"  # write models/c2_srcd.pt at the end
MODEL_OUT = Path(os.environ.get("SWARA_C2_MODEL_OUT", ROOT / "models" / "c2_srcd.pt"))
REPORT_DIR = Path(os.environ.get("SWARA_C2_REPORT_DIR", ROOT / "training" / "reports"))
NOTE = os.environ.get("SWARA_C2_NOTE", "")
MIN_PRECISION = float(os.environ.get("SWARA_C2_MIN_PRECISION", 0.5))
print("data:", DATA, "| folds:", FOLDS, "| epochs:", EPOCHS)

data: D:\Swara-RP\server\vision-service\data\casa.npz | folds: 5 | epochs: 60


## 1. Load the dataset

In [2]:
d = np.load(DATA, allow_pickle=False)
required = {"X", "X_raw", "W", "rules_pred", "srcd_pred", "y", "groups"}
missing = required - set(d.files)
assert not missing, f"{DATA.name} was built by an older build_dataset.py (missing {sorted(missing)}); rebuild it."

X, X_raw, W = d["X"], d["X_raw"], d["W"].astype(float)
y, groups = d["y"], d["groups"]
rules_pred, srcd_pred = d["rules_pred"], d["srcd_pred"]
CATS = ("FG", "HM")
print(f"{len(y)} moments, {len(set(groups))} participants")
pd.DataFrame({"positives": y.sum(axis=0), "share": y.mean(axis=0).round(3)}, index=CATS)

2062 moments, 24 participants


,positives,share
FG,1458,0.707
HM,165,0.080


## 2. Participant-grouped folds

In [3]:
from sklearn.model_selection import GroupKFold

n_splits = min(FOLDS, len(set(groups)))
assert n_splits >= 2, "Need at least two participants."
folds = list(GroupKFold(n_splits=n_splits).split(X, y, groups))
for k, (tr, te) in enumerate(folds):
    assert not set(groups[tr]) & set(groups[te])
    print(f"fold {k}: train {len(tr)} moments / {len(set(groups[tr]))} participants, test {len(te)} / {len(set(groups[te]))}")

fold 0: train 1647 moments / 22 participants, test 415 / 2
fold 1: train 1649 moments / 19 participants, test 413 / 5
fold 2: train 1652 moments / 19 participants, test 410 / 5
fold 3: train 1651 moments / 18 participants, test 411 / 6
fold 4: train 1649 moments / 18 participants, test 413 / 6


In [4]:
from training.train_srcd import assessed, class_metrics, summarise, tune_thresholds, validation_split

def score_fold(tr, te, pred, scores=None):
    return class_metrics(y[te], pred, scores, y_train=y[tr])

def apply(scores, thr):
    return np.stack([scores[:, i] >= thr[c] for i, c in enumerate(CATS)], axis=1).astype(int)

def inner_split(tr, k):
    # Participant-grouped validation split of this training fold, for threshold tuning.
    a, b = validation_split(groups[tr], SEED + k)
    return tr[a], tr[b]

results = {}     # approach -> list of per-fold metric dicts
thresholds = {}  # approach -> list of per-fold tuned thresholds

## 3. Approaches without training: rules and label-free SRCD

In [5]:
majority = lambda tr, n: np.tile(np.round(y[tr].mean(axis=0)).astype(int), (n, 1))
results["0 majority class"] = [score_fold(tr, te, majority(tr, len(te))) for tr, te in folds]
for name, pred in [("1 rules", rules_pred), ("4a SRCD label-free", srcd_pred)]:
    results[name] = [score_fold(tr, te, pred[te]) for tr, te in folds]

## 4. Random Forest on 2-second windows

Statistics of each raw feature over a 2-s window centred on the moment (mean, SD, range, mean absolute change, face ratio): the earlier prototype's input. Missing values (no face) take the training fold's median; positives are weighted up (balanced classes).

In [6]:
from sklearn.ensemble import RandomForestClassifier

def rf_scores(fit_idx, idx):
    med = np.nanmedian(W[fit_idx], axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    fill = lambda A: np.where(np.isnan(A), med, A)
    scores = np.zeros((len(idx), 2))
    for i in range(2):
        if len(set(y[fit_idx, i])) < 2:  # a fold without positives can't be learned
            continue
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=SEED, n_jobs=-1)
        rf.fit(fill(W[fit_idx]), y[fit_idx, i])
        scores[:, i] = rf.predict_proba(fill(W[idx]))[:, 1]
    return scores

name = "2 Random Forest (2-s windows)"
results[name], thresholds[name] = [], []
for k, (tr, te) in enumerate(folds):
    fit_idx, val_idx = inner_split(tr, k)
    both = rf_scores(fit_idx, np.concatenate([val_idx, te]))
    thr = tune_thresholds(both[:len(val_idx)], y[val_idx])
    s_te = both[len(val_idx):]
    results[name].append(score_fold(tr, te, apply(s_te, thr), s_te))
    thresholds[name].append(thr)

## 5. GRU without and with the self-comparison step

The same small GRU and the same training (`training/train_srcd.py`), on two inputs:
- **without self-comparison:** raw feature values standardised with the *training fold's population* mean and SD;
- **SRCD:** each moment's deviation from the same person's fluent baseline (what the service computes).

If SRCD beats the population-standardised version, the self-comparison idea itself is what helps.

In [7]:
import torch
from training.train_srcd import fit, predict

torch.set_num_threads(max(1, os.cpu_count() - 1))

def population_standardise(tr, te):
    flat = X_raw[tr].reshape(-1, X_raw.shape[2])
    mu, sd = np.nanmean(flat, axis=0), np.nanstd(flat, axis=0)
    mu, sd = np.where(np.isnan(mu), 0, mu), np.where(~(sd > 1e-6), 1, sd)
    scale = lambda A: np.nan_to_num((A - mu) / sd / 10.0, nan=0.0).astype(np.float32)  # same ±1 range as SRCD input
    out_tr, out_te = scale(X_raw[tr]), scale(X_raw[te])
    out_tr[..., -1], out_te[..., -1] = np.nan_to_num(X_raw[tr][..., -1]), np.nan_to_num(X_raw[te][..., -1])  # face channel stays 0/1
    return out_tr, out_te

def gru_fold(name, A, tr, te, k):
    # A: inputs for every moment (already scaled for this fold). Train on the fold's training participants minus a
    # validation group, tune each class's threshold there, then score the test fold.
    fit_idx, val_idx = inner_split(tr, k)
    model = fit("gru", A[fit_idx], y[fit_idx], EPOCHS, 3e-3, SEED + k)
    thr = tune_thresholds(predict(model, A[val_idx]), y[val_idx])
    s_te = predict(model, A[te])
    results[name].append(score_fold(tr, te, apply(s_te, thr), s_te))
    thresholds[name].append(thr)

t0 = time.time()
for name in ("3 GRU without self-comparison", "4b SRCD + GRU"):
    results[name], thresholds[name] = [], []
for k, (tr, te) in enumerate(folds):
    a, b = population_standardise(tr, te)
    pop = np.zeros_like(X_raw, dtype=np.float32); pop[tr], pop[te] = a, b
    gru_fold("3 GRU without self-comparison", pop, tr, te, k)
    gru_fold("4b SRCD + GRU", X, tr, te, k)
print(f"GRU folds: {time.time() - t0:.0f} s")

GRU folds: 828 s


## 6. Results

In [17]:
CASA_REFERENCE = {"FG": 0.63, "HM": 0.28}  # CASA (Interspeech 2025), best video model; their own test set

summaries = {name: summarise(per_fold) for name, per_fold in results.items()}
fmt = lambda v: "n/a" if v is None else f"{v['mean']:.2f} ± {v['sd']:.2f}"

rows = []
for name, sm in summaries.items():
    row = {"approach": name}
    for c in CATS:
        row[f"{c} acc"] = fmt(sm[c]["accuracy"])
        row[f"{c} majority acc"] = fmt(sm[c]["majorityAccuracy"])
        row[f"{c} precision"] = fmt(sm[c]["precision"])
        row[f"{c} recall"] = fmt(sm[c]["recall"])
        row[f"{c} F1"] = fmt(sm[c]["f1"])
        row[f"{c} PR-AUC"] = fmt(sm[c]["prAuc"])
    row["macro F1"] = fmt(sm["macro"]["f1"])
    rows.append(row)
rows.append({"approach": "5 CASA published video model (reference, other test set)",
             "FG F1": f"{CASA_REFERENCE['FG']:.2f}", "HM F1": f"{CASA_REFERENCE['HM']:.2f}",
             "macro F1": f"{np.mean(list(CASA_REFERENCE.values())):.3f}"})
table = pd.DataFrame(rows).set_index("approach").fillna("")
table.T  # one column per approach

approach,0 majority class,1 rules,4a SRCD label-free,2 Random Forest (2-s windows),3 GRU without self-comparison,4b SRCD + GRU,"5 CASA published video model (reference, other test set)"
FG acc,0.71 ± 0.11,0.62 ± 0.09,0.33 ± 0.08,0.71 ± 0.11,0.71 ± 0.11,0.70 ± 0.11,
FG majority acc,0.71 ± 0.11,0.71 ± 0.11,0.71 ± 0.11,0.71 ± 0.11,0.71 ± 0.11,0.71 ± 0.11,
FG precision,0.71 ± 0.11,0.73 ± 0.12,0.79 ± 0.22,0.71 ± 0.11,0.71 ± 0.11,0.71 ± 0.11,
FG recall,1.00 ± 0.00,0.71 ± 0.09,0.06 ± 0.02,1.00 ± 0.00,0.99 ± 0.02,0.99 ± 0.01,
FG F1,0.82 ± 0.08,0.72 ± 0.09,0.11 ± 0.04,0.82 ± 0.08,0.82 ± 0.08,0.82 ± 0.08,0.63
FG PR-AUC,n/a,n/a,n/a,0.75 ± 0.10,0.72 ± 0.16,0.74 ± 0.15,
HM acc,0.92 ± 0.07,0.83 ± 0.11,0.91 ± 0.06,0.79 ± 0.19,0.81 ± 0.15,0.78 ± 0.11,
HM majority acc,0.92 ± 0.07,0.92 ± 0.07,0.92 ± 0.07,0.92 ± 0.07,0.92 ± 0.07,0.92 ± 0.07,
HM precision,0.00 ± 0.00,0.16 ± 0.16,0.28 ± 0.25,0.16 ± 0.17,0.10 ± 0.10,0.13 ± 0.11,
HM recall,0.00 ± 0.00,0.42 ± 0.36,0.29 ± 0.30,0.34 ± 0.32,0.22 ± 0.26,0.30 ± 0.18,


In [9]:
# Which classes each trained approach may decide: precision >= MIN_PRECISION and accuracy above the majority class.
status = pd.DataFrame({name: {c: "assessed" if ok else "NOT assessed" for c, ok in assessed(summaries[name], MIN_PRECISION).items()}
                       for name in thresholds}).T
status

,FG,HM
2 Random Forest (2-s windows),NOT assessed,NOT assessed
3 GRU without self-comparison,assessed,NOT assessed
4b SRCD + GRU,NOT assessed,NOT assessed


In [10]:
# Paired by fold: SRCD + GRU minus each other approach (macro F1). Positive in most folds = a consistent gain.
def paired(a, b):
    diff = np.array([fa["macro"]["f1"] - fb["macro"]["f1"] for fa, fb in zip(results[a], results[b])])
    return f"{diff.mean():+.3f} (better in {int((diff > 0).sum())} of {len(diff)} folds)"

pd.Series({other: paired("4b SRCD + GRU", other) for other in results if other != "4b SRCD + GRU"},
          name="SRCD + GRU minus ... (macro F1)")

0 majority class                 +0.065 (better in 5 of 5 folds)
1 rules                          +0.003 (better in 2 of 5 folds)
4a SRCD label-free               +0.290 (better in 5 of 5 folds)
2 Random Forest (2-s windows)    -0.033 (better in 2 of 5 folds)
3 GRU without self-comparison    +0.004 (better in 3 of 5 folds)
Name: SRCD + GRU minus ... (macro F1), dtype: object

**How to read it** (the plan's test):
- Every method's accuracy must beat the majority class (0); a high accuracy with recall near 0 means it learned "always no".
- SRCD (4a or 4b) above the rules (1) and the Random Forest (2) → it beats the prototype's methods.
- 3 (without self-comparison) clearly below 4b (with it) → the speaker-relative step is what helps.
- CASA's numbers come from a different split and model, so they are only a scale for "respectable"; the plan notes that even modest numbers are acceptable.

## 7. Save the report and train the final model

In [11]:
from datetime import datetime, timezone

REPORT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d")
record = {
    "data": DATA.name, "moments": int(len(y)), "participants": int(len(set(groups))), "folds": n_splits,
    "epochs": EPOCHS, "seed": SEED, "note": NOTE,
    "minPrecision": MIN_PRECISION, "results": results, "summaries": summaries, "thresholds": thresholds,
    "assessed": {name: assessed(summaries[name], MIN_PRECISION) for name in thresholds}, "casaReference": CASA_REFERENCE,
}
(REPORT_DIR / f"c2-comparison-{stamp}.json").write_text(json.dumps(record, indent=1, default=float))
(REPORT_DIR / f"c2-comparison-{stamp}.md").write_text(
    f"# C2 five-way comparison ({stamp})\n\n{len(y)} moments, {len(set(groups))} participants, {n_splits} participant-grouped folds. {NOTE}\n\n"
    + table.T.to_markdown() + "\n\n## Classes each trained model may decide\n\n" + status.to_markdown() + "\n")
print("report written to", REPORT_DIR)

report written to D:\Swara-RP\server\vision-service\training\reports


In [12]:
# The service loads this file when C2_DETECTOR=gru (app/detection/learned.py).
from training.train_srcd import save_checkpoint

if SAVE_MODEL:
    final = fit("gru", X, y, EPOCHS, 3e-3, SEED)
    best = "4b SRCD + GRU"
    thr = {c: float(np.median([t[c] for t in thresholds[best]])) for c in CATS}  # the folds' validation-tuned thresholds
    ok = assessed(summaries[best], MIN_PRECISION)
    save_checkpoint(final, "gru", X, y, groups, MODEL_OUT, data_name=DATA.name, epochs=EPOCHS, cv=summaries[best],
                    note=NOTE, thresholds=thr, assessed_classes=ok)
    print("saved", MODEL_OUT, "| thresholds", thr, "| assessed", ok)

saved D:\Swara-RP\server\vision-service\models\c2_srcd.pt | thresholds {'FG': 0.05, 'HM': 0.125} | assessed {'FG': False, 'HM': False}


In [13]:
# Check that the service accepts the checkpoint.
if SAVE_MODEL:
    from app.detection.learned import LearnedSRCD
    det = LearnedSRCD("gru", MODEL_OUT)
    print("service loads it:", det.available(), "| classes the model decides:", det.assessed())

service loads it: (True, None) | classes the model decides: {'FG': False, 'HM': False}
